In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df=pd.read_csv('dataset.csv')
print(df.head())

                                                text  label
0  Gere faults Trump for blurring meaning of 'ref...      1
1  German parties start to find common ground in ...      1
2  Senate Democratic leader says Attorney General...      1
3  Tennis: Kyrgios fined $10,000 for Shanghai wal...      1
4   Trump Threw Mar-A-Lago Fundraiser For Woman A...      0


In [4]:
df.shape

(45757, 2)

In [5]:
df.columns

Index(['text', 'label'], dtype='object')

In [8]:
df.isnull().sum()   

text     0
label    0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
import re
import string
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    
    cleaned_text = " ".join(tokens)
    
    return cleaned_text

In [11]:
df.head()

,text,label
0,Gere faults Trump for blurring meaning of 'ref...,1
1,German parties start to find common ground in ...,1
2,Senate Democratic leader says Attorney General...,1
3,"Tennis: Kyrgios fined $10,000 for Shanghai wal...",1
4,Trump Threw Mar-A-Lago Fundraiser For Woman A...,0


In [12]:
df['cleaned_text'] = df['text'].apply(preprocess_text)

In [13]:
df['cleaned_text'].head()

0    gere faults trump blurring meaning refugee ter...
1    german parties start find common ground coalit...
2    senate democratic leader says attorney general...
3    tennis kyrgios fined 10000 shanghai walk tenni...
4    trump threw maralago fundraiser woman center b...
Name: cleaned_text, dtype: object

In [14]:
df.columns

Index(['text', 'label', 'cleaned_text'], dtype='object')

In [15]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
print(X_train.shape)
print(X_test.shape)

(36605,)
(9152,)


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train)    
X_test_tfidf = tfidf.transform(X_test)


In [18]:
print(X_train_tfidf.shape)
print(X_test_tfidf.shape)

(36605, 227975)
(9152, 227975)


Logistic Regression 

In [19]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)
print(y_pred[:10])


[1 0 1 1 0 0 1 0 1 1]


Evaluation 

In [20]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.9731


In [21]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('model', LogisticRegression(max_iter=1000))
])

scores = cross_val_score(
    pipeline,
    df['cleaned_text'],
    df['label'],
    cv=5,
    scoring='accuracy'
)

print(scores)
print("Mean Accuracy:", scores.mean())
print("Standard Deviation:", scores.std())

[0.97344843 0.97650787 0.97235275 0.97792591 0.97278986]
Mean Accuracy: 0.9746049621616499
Standard Deviation: 0.002207006170892135


In [23]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")  
print(conf_matrix)
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[4463  109]
 [ 137 4443]]
              precision    recall  f1-score   support

           0       0.97      0.98      0.97      4572
           1       0.98      0.97      0.97      4580

    accuracy                           0.97      9152
   macro avg       0.97      0.97      0.97      9152
weighted avg       0.97      0.97      0.97      9152

